# Workflows: Testing Cromwell and Nextflow in *All of Us*

**A test notebook (November 2026).** The *Running Workflows* class teaches **dsub**, which we tested step by step. Its last part shows short **Cromwell** (WDL) and **Nextflow** examples. This notebook runs them, small and cheap, and **records what happens**: what worked, what failed, how long each step took, and how we fixed it.

✅ **Tested November 2026** (Cromwell 92, Nextflow 25.10.2): every test worked once fixed, in 2 to 4 minutes each, and every PLINK result matched the app exactly. The fixes are built in and marked in the configuration files. The results and lessons are on the class page: *Running Workflows* → *Tested: Cromwell and Nextflow*. Rerun this notebook to check that the settings still work (for example after a new Cloud SDK or Cromwell version).

| Part | Test | Where it runs | Rough cost |
|---|---|---|---|
| 1 | Setup: tools, settings, and a test log | the app | free |
| 2 | Cromwell: *hello world*, then PLINK on chromosome 21 | app terminal (*run mode*) | cents |
| 3 | Cromwell: the same WDL from the **Workflows** tab | the Workbench web page (*server mode*) | cents |
| 4 | Nextflow: *hello*, then PLINK on chromosomes 19 to 22 | the app | cents |
| 5 | The test log: what to report back | the app | free |

**No participant data is printed.** The PLINK test computes **allele frequencies** of common ClinVar variants across everyone in the files; we look only at the **number of variants** and how closely two runs agree.

⏱️ **Time:** about 45 to 60 minutes, mostly waiting for cloud machines. 💲 **Cost:** a few cents of job machines, plus the app.

## Part 1. Setup

### Tools and settings

Cromwell and Nextflow are **installed in the app**. Cromwell is a Java program (`$CROMWELL_JAR`); its helper, **womtool**, checks WDL files for mistakes before you run them.

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

# Which settings does the app give us? (we print only whether each is set, not its value)
env_vars <- c("GOOGLE_CLOUD_PROJECT", "GOOGLE_PROJECT", "PET_SA_EMAIL", "CROMWELL_JAR", "WORKSPACE_BUCKET")
tibble(variable = env_vars, set = Sys.getenv(env_vars) != "")

**🔍 Read the output:** `TRUE` or `FALSE` for each setting. The *All of Us* tutorials use `GOOGLE_PROJECT`, `PET_SA_EMAIL`, and `WORKSPACE_BUCKET`, which come from their own setup notebooks; **we don't rely on them**. We use `GOOGLE_CLOUD_PROJECT` (as in all our notebooks), the account from `gcloud`, and the bucket address from the Workbench CLI, as in the dsub class.

In [ ]:
CROMWELL_JAR <- Sys.getenv("CROMWELL_JAR")
WOMTOOL_JAR  <- file.path(dirname(CROMWELL_JAR), "womtool.jar")   # next to Cromwell, per the tutorial

c(cromwell_jar = file.exists(CROMWELL_JAR), womtool_jar = file.exists(WOMTOOL_JAR))
# intern = TRUE: without it, the output goes to the app's log, not the notebook
system("java -version 2>&1 | head -1", intern = TRUE)
system(paste("java -jar", shQuote(CROMWELL_JAR), "--version"), intern = TRUE)
system("nextflow -v", intern = TRUE)

**🔍 Read the output:** `TRUE` twice (both files found), then the **versions** of Java, Cromwell, and Nextflow. Write the versions down: the class page reports them. If `womtool_jar` is `FALSE`, look for it with `list.files(dirname(CROMWELL_JAR))`.

### The account, the bucket, and the *All of Us* mirror

The same settings as the dsub class: jobs run as **your account**, write to **your bucket** (its `gs://` address), and get their software **images** from the *All of Us* mirror of Docker Hub, because job machines have no internet access.

In [ ]:
SERVICE_ACCOUNT <- system("gcloud config get-value account", intern = TRUE)

# The bucket's gs:// address, from the Workbench CLI (as in the dsub class)
BUCKET_GS <- local({
  name <- basename(BUCKET_NAME)
  found <- suppressWarnings(system2("wb", c("resource", "resolve", "--name", shQuote(name)),
                                    stdout = TRUE, stderr = FALSE))
  found <- str_subset(found, "^gs://")
  if (length(found) == 0) found <- paste0("gs://", name)
  str_remove(found[1], "/$")
})

WORK_GS <- paste0(BUCKET_GS, "/data/workflows")   # where job files and results go

MIRROR       <- "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
PLINK_IMAGE  <- paste0(MIRROR, "/biocontainer/plink2:alpha2.3_jan2020")   # as in the dsub class
UBUNTU_IMAGE <- paste0(MIRROR, "/ubuntu:24.04")                           # a pinned version

CLINVAR_GS  <- "gs://vwb-aou-datasets-controlled/v9/wgs/short_read/snpindel/clinvar/plink_bed"
CLINVAR_DIR <- file.path(aou_data_dir("v9"), "wgs", "short_read", "snpindel", "clinvar", "plink_bed")

# Local working folders on the app's disk (not the bucket: both tools write many small files)
TEST_DIR <- file.path(Sys.getenv("HOME"), "workflow_tests")
dir.create(file.path(TEST_DIR, "cromwell"), recursive = TRUE, showWarnings = FALSE)
dir.create(file.path(TEST_DIR, "nextflow"), recursive = TRUE, showWarnings = FALSE)

system2("gcloud", c("storage", "ls", paste0(WORK_GS, "/")), stdout = TRUE) %>% head(5)

**🔍 Read the output:** a few folder names in `data/workflows/` (from the dsub class, if you ran it), or an error. An error means the bucket address is wrong: set it by hand, e.g. `BUCKET_GS <- "gs://my-bucket-name"`, then rerun the line that sets `WORK_GS`.

### A test log

Every test adds one row to `test_log`: what we ran, whether it **worked**, how many **minutes** it took, and a **note** (the error, or what we changed). At the end, this table is what to report back.

In [ ]:
test_log <- tibble(test = character(), tool = character(), result = character(),
                   minutes = numeric(), note = character())

log_test <- function(test, tool, result, minutes = NA, note = "") {
  test_log <<- bind_rows(test_log %>% filter(.data$test != !!test),   # rerunning a test replaces its row
                         tibble(test = test, tool = tool, result = result,
                                minutes = round(minutes, 1), note = note))
  test_log
}

# Show the lines of a log that most often explain a failure
error_lines <- function(lines, n = 15) {
  hits <- str_subset(lines, regex("error|exception|failed|denied|not found|no such", ignore_case = TRUE))
  head(unique(str_trunc(str_remove(hits, "^(\\[[^]]*\\] )+"), 600)), n)   # drop the long timestamp prefix
}

### A reference result: PLINK on chromosome 21, in the app

Both tools will compute the **allele frequencies** of the common variants on chromosome 21 (`plink2 --maf 0.01 --freq`). We run the same command **in the app** first, so we can check that the workflows give the same answer. It takes a minute or two.

In [ ]:
APP_OUT <- file.path(TEST_DIR, "app_chr21")
dir.create(APP_OUT, showWarnings = FALSE)

app_seconds <- system.time(
  app_run <- system2("plink2", c("--bfile", shQuote(file.path(CLINVAR_DIR, "clinvar.chr21")),
                                 "--maf", "0.01", "--freq", "--out", shQuote(file.path(APP_OUT, "chr21"))),
                     stdout = TRUE, stderr = TRUE)
)[["elapsed"]]

# Read a plink2 .afreq file: we keep the variant ID and the frequency of the ALT allele
read_afreq <- function(file) {
  read_tsv(file, col_types = cols(.default = "c")) %>%
    transmute(ID, ALT, ALT_FREQS = suppressWarnings(as.numeric(ALT_FREQS)))
}

app_freq <- read_afreq(file.path(APP_OUT, "chr21.afreq"))
c(app_minutes = round(app_seconds / 60, 1), variants = nrow(app_freq))

**🔍 Read the output:** the minutes it took, and the number of common variants on chromosome 21. (If `plink2` isn't found, run **4.1** first: it shows how PLINK is set up in the app.)

## Part 2. Cromwell in the app (*run mode*)

**Run mode:** `java -jar $CROMWELL_JAR run …` starts Cromwell **in the app**, sends the workflow's tasks to **Google Batch** (each on its own machine), waits until they finish, and stops. The app must **stay on** until then.

Cromwell needs a **configuration file** with the *All of Us* settings. Ours follows the *All of Us* tutorial, with three changes, marked in the comments.

⚠️ **Change 3 is essential (November 2026).** To copy files on and off job machines, Cromwell uses a Google Cloud SDK **helper image**. Version **592** of that image (November 3, 2026) removed a program it relies on, so every task failed within minutes, with only Cromwell's scripts in the task folder and the message *"The job was stopped before the command finished"*. Pinning version **591** fixes it: the same problem, and the same fix, as `DSUB_CLOUD_SDK_IMAGE` in the dsub class.

In [ ]:
cromwell_conf <- file.path(TEST_DIR, "cromwell", "cromwell.conf")

write_lines(glue::glue('
backend {{
  default = "GCPBATCH"
  providers {{
    Local.config.root = "/dev/null"   # no jobs in the app itself
    GCPBATCH {{
      actor-factory = "cromwell.backend.google.batch.GcpBatchBackendLifecycleActorFactory"
      config {{
        project = "{BILLING_PROJECT}"
        concurrent-job-limit = 10
        root = "{WORK_GS}/cromwell-executions"   # work files and results
        virtual-private-cloud {{
          network-name    = "projects/{BILLING_PROJECT}/global/networks/network"
          subnetwork-name = "projects/{BILLING_PROJECT}/regions/*/subnetworks/subnetwork"
        }}
        batch {{
          auth = "application-default"
          compute-service-account = "{SERVICE_ACCOUNT}"   # change 1: from gcloud, not PET_SA_EMAIL
          location = "us-central1"
        }}
        default-runtime-attributes {{
          noAddress: true   # no public internet address (required)
        }}
        filesystems {{
          gcs {{
            auth = "application-default"
            project = "{BILLING_PROJECT}"   # change 2: who pays to read requester-pays buckets
          }}
        }}
      }}
    }}
  }}
}}

# change 3: pin the file-copying helper image that Cromwell uses (version 592 breaks it)
google.cloud-sdk-image-url = "gcr.io/google.com/cloudsdktool/google-cloud-cli:591.0.0-slim"
'), cromwell_conf)

cat(str_replace(read_lines(cromwell_conf), SERVICE_ACCOUNT, "<your account>"), sep = "\n")

**🔍 Read the output:** the configuration file, with your account hidden. The settings match the table on the class page: project, work folder (`root`), network, account, region, and **no public address**.

### A helper to validate, run, and time a WDL workflow

`run_cromwell()` does what the tutorial does in many cells: **validate** the WDL with womtool, **run** it, save Cromwell's messages to a log file, and find the **status**, the **workflow ID**, and the **outputs** in that log.

In [ ]:
run_cromwell <- function(wdl, inputs_json, name) {
  old <- setwd(file.path(TEST_DIR, "cromwell")); on.exit(setwd(old))
  check <- system2("java", c("-jar", shQuote(WOMTOOL_JAR), "validate", wdl, "-i", inputs_json),
                   stdout = TRUE, stderr = TRUE)
  message("womtool: ", paste(check, collapse = " "))
  if (!is.null(attr(check, "status"))) return(invisible(list(status = "Invalid WDL", minutes = 0, log = check)))

  log_file <- paste0(name, ".log")
  seconds <- system.time(
    out <- suppressWarnings(system2("java", c(paste0("-Dconfig.file=", cromwell_conf), "-jar",
                                              shQuote(CROMWELL_JAR), "run", wdl, "-i", inputs_json,
                                              "-m", paste0(name, "_metadata.json")),
                                    stdout = TRUE, stderr = TRUE))
  )[["elapsed"]]
  write_lines(out, log_file)

  final  <- str_match(out, "Workflow actor for ([a-f0-9-]+) completed with status '(\\w+)'")
  final  <- final[!is.na(final[, 1]), , drop = FALSE]
  status <- if (nrow(final) > 0) final[1, 3] else "Unknown (read the log)"
  outputs <- unique(str_extract(out, "gs://[^\" ]+"))
  outputs <- outputs[!is.na(outputs) & str_detect(outputs, "call-")]

  message("Status: ", status, "   Minutes: ", round(seconds / 60, 1), "   Log: ", log_file)
  if (status != "Succeeded") print(error_lines(out))
  invisible(list(status = status, id = if (nrow(final) > 0) final[1, 2] else NA,
                 minutes = seconds / 60, outputs = outputs, log = out))
}

### Test C1: *hello world*

The smallest possible workflow: one task that prints "hello world". If this works, the **configuration** is right. Unlike the tutorial, the image comes from the *All of Us* **mirror** (`UBUNTU_IMAGE`). This takes a few minutes, mostly to start the machine.

In [ ]:
write_lines(glue::glue('
version 1.0

workflow hello_world {{
  call say_hello
}}

task say_hello {{
  command {{
    echo "hello world"
  }}
  output {{
    String greeting = read_string(stdout())
  }}
  runtime {{
    docker: "{UBUNTU_IMAGE}"
    machineType: "n2-standard-2"
  }}
}}
'), file.path(TEST_DIR, "cromwell", "hello_world.wdl"))
write_lines("{}", file.path(TEST_DIR, "cromwell", "hello_world.json"))

c1 <- run_cromwell("hello_world.wdl", "hello_world.json", "c1_hello")
str_subset(c1$log, "hello_world.say_hello.greeting|greeting")
log_test("C1 Cromwell hello world (mirror image)", "Cromwell run mode", c1$status, c1$minutes)

**🔍 Read the output:** `Status: Succeeded`, the minutes it took, and the output line with `"hello world"`. The test log now has one row.

If it **failed**, the lines printed after the status are the likely cause. Common ones:

| Message | Likely cause | Try |
|---|---|---|
| *The job was stopped before the command finished*; the task folder has only `gcs_*.sh` and `script` (no `rc`, `stdout`, `stderr`) | Cromwell's helper image is broken (Google Cloud SDK 592) | check that the config has change 3 (`google.cloud-sdk-image-url`, pinned to 591) |
| `MANIFEST_UNKNOWN` … `falling back to OCI manifest` | Cromwell couldn't fingerprint the image (used only to reuse earlier results) | nothing: a **warning**, not the cause of a failure |
| `PERMISSION_DENIED` … `serviceAccount` | the account can't run jobs | check `SERVICE_ACCOUNT`; try the tutorial's `PET_SA_EMAIL` if it's set |
| an error about the **network** or **subnetwork** | the network names | try `regions/us-central1/subnetworks/subnetwork` instead of `regions/*/…` |
| the image can't be **pulled** | the image address | check `UBUNTU_IMAGE` |

Add what you saw and what you changed to the log, e.g. `log_test("C1 …", "Cromwell run mode", "Succeeded", c1$minutes, "needed …")`.

### Test C2: PLINK on chromosome 21

The WDL from the class page: PLINK on one chromosome. Cromwell **copies** the three PLINK files onto the job's machine (like dsub), so the task needs a **disk** big enough for them. We ask for an **SSD** disk, which was much faster to copy onto in the dsub class.

In [ ]:
write_lines('
version 1.0

workflow plink_freq {
  input {
    File bed
    File bim
    File fam
    String docker_image
  }
  call plink_check { input: bed = bed, bim = bim, fam = fam, docker_image = docker_image }
  output {
    File freq = plink_check.freq
  }
}

task plink_check {
  input {
    File bed
    File bim
    File fam
    String docker_image
  }
  command {
    plink2 --bed ~{bed} --bim ~{bim} --fam ~{fam} --maf 0.01 --freq --out chr
  }
  output {
    File freq = "chr.afreq"
  }
  runtime {
    docker: docker_image
    machineType: "n2-standard-2"
    disks: "local-disk 30 SSD"
  }
}
', file.path(TEST_DIR, "cromwell", "plink_freq.wdl"))

plink_inputs <- list(
  plink_freq.bed = paste0(CLINVAR_GS, "/clinvar.chr21.bed"),
  plink_freq.bim = paste0(CLINVAR_GS, "/clinvar.chr21.bim"),
  plink_freq.fam = paste0(CLINVAR_GS, "/clinvar.chr21.fam"),
  plink_freq.docker_image = PLINK_IMAGE
)
jsonlite::write_json(plink_inputs, file.path(TEST_DIR, "cromwell", "plink_freq.json"),
                     auto_unbox = TRUE, pretty = TRUE)

c2 <- run_cromwell("plink_freq.wdl", "plink_freq.json", "c2_plink_chr21")
c2$outputs

**🔍 Read the output:** the status, the minutes, and the `gs://` address of the results file (`…/call-plink_check/…/chr.afreq`). Now compare it with the app's result:

In [ ]:
compare_freq <- function(gs_file, label) {
  if (length(gs_file) == 0 || is.na(gs_file)) {
    message("No results file: the workflow didn't finish. Fix the failure above first.")
    return(c(variants_app = nrow(app_freq), variants_job = NA, matched = NA, max_freq_difference = NA))
  }
  local_copy <- file.path(TEST_DIR, paste0(label, ".afreq"))
  system2("gcloud", c("storage", "cp", gs_file, local_copy), stdout = FALSE, stderr = FALSE)
  job_freq <- read_afreq(local_copy)
  both <- inner_join(app_freq, job_freq, by = c("ID", "ALT"), suffix = c("_app", "_job"))
  c(variants_app = nrow(app_freq), variants_job = nrow(job_freq), matched = nrow(both),
    max_freq_difference = max(abs(both$ALT_FREQS_app - both$ALT_FREQS_job), na.rm = TRUE))
}

c2_check <- compare_freq(str_subset(c2$outputs, "chr\\.afreq$")[1], "cromwell_chr21")
c2_check
log_test("C2 Cromwell PLINK chr21 (mirror image, SSD)", "Cromwell run mode", c2$status, c2$minutes,
         paste0("matched ", c2_check[["matched"]], "/", c2_check[["variants_app"]],
                " variants; max difference ", signif(c2_check[["max_freq_difference"]], 2)))

**🔍 Read the output:** the same number of variants in the app and the job, all **matched**, and a maximum difference close to **0** (tiny differences can come from the different PLINK versions).

### How long did each step take?

Cromwell keeps a timeline of each task in its **metadata**, which `run_cromwell()` saves as a JSON file (`-m`). Its log doesn't record these steps, so we read the metadata.

In [ ]:
meta <- jsonlite::read_json(file.path(TEST_DIR, "cromwell", "c2_plink_chr21_metadata.json"))

c2_steps <- map_dfr(names(meta$calls), function(task)
  map_dfr(meta$calls[[task]][[1]]$executionEvents, function(e)
    tibble(task = task, step = e$description, start = ymd_hms(e$startTime), end = ymd_hms(e$endTime)))) %>%
  arrange(start) %>%
  mutate(seconds = round(as.numeric(difftime(end, start, units = "secs"))))
c2_steps %>% select(task, step, seconds)

**🔍 Read the output:** one row per step of the task, with its **seconds**. Compare the waiting and set-up steps (getting a machine, copying the files) with the step that runs PLINK. (If the file isn't found, rerun C2: the metadata file was added after our first test.)

### Test C3: an image straight from Docker Hub

The *All of Us* tutorial uses `ubuntu:latest` and `biocontainer/plink2:…` **without** the mirror. In the dsub class, images from Docker Hub **failed**, because job machines have no public address. Does Cromwell behave the same? This tells us whether the class must insist on the mirror for Cromwell too.

⚠️ If it fails, it may take **several minutes** to give up. You can stop the cell (■) after 10 minutes and record that.

In [ ]:
write_lines(str_replace(read_lines(file.path(TEST_DIR, "cromwell", "hello_world.wdl")),
                        fixed(UBUNTU_IMAGE), "ubuntu:latest"),
            file.path(TEST_DIR, "cromwell", "hello_dockerhub.wdl"))

c3 <- run_cromwell("hello_dockerhub.wdl", "hello_world.json", "c3_dockerhub")
log_test("C3 Cromwell hello world (Docker Hub image)", "Cromwell run mode", c3$status, c3$minutes,
         paste(head(error_lines(c3$log), 2), collapse = " | "))

**🔍 Read the output:** `Succeeded` or `Failed`. Either way, it's a useful answer:

- **Failed**, with a message about pulling the image: the class must say "**always use the mirror**" for Cromwell, as for dsub.
- **Succeeded**: Cromwell (or Google Batch) reaches Docker Hub in some other way. The mirror is still the safer choice, because it's what *All of Us* supports.

## Part 3. Cromwell from the Workflows tab (*server mode*)

On the **Workflows** tab, the Workbench runs Cromwell **for you** (*server mode*): you don't need an app running, and you can watch the workflow's progress on the page. This is the route many students will try first.

This part is **done by hand** on the web page. The next cell copies the WDL and the inputs into your bucket, where the Workflows tab can find them.

In [ ]:
TAB_GS <- paste0(WORK_GS, "/workflows_tab")
system2("gcloud", c("storage", "cp",
                    file.path(TEST_DIR, "cromwell", c("plink_freq.wdl", "plink_freq.json")),
                    paste0(TAB_GS, "/")))
system2("gcloud", c("storage", "ls", paste0(TAB_GS, "/")), stdout = TRUE)
cat(read_lines(file.path(TEST_DIR, "cromwell", "plink_freq.json")), sep = "\n")   # the values for the form

**🔍 Read the output:** two files in `data/workflows/workflows_tab/` (the WDL and its inputs), then the four input values you'll copy into the form.

### Steps on the Workflows tab

What we found in November 2026 (it took about 4 minutes, and matched the app):

1. **Workflows** tab → **Setup** → **+ Add workflow** → **Workspace bucket**. The bucket list may open on `dataproc-staging-…` (*"0 workflows found"*): choose **your workspace bucket**. Select `data/workflows/workflows_tab/plink_freq.wdl` → **Next** → **Add to workspace**.
2. Select `plink_freq` → **+ New job** → *Run a single job* → **Next**.
3. **Inputs:** a **form**, one box per input (no JSON upload). Copy the four values printed by the cell above, **without** quotation marks. Keep the run options as they are.
4. **Outputs:** your workspace bucket; the folder is named after the job. **Submit job**.
5. Follow it on **Job status** (*Starting* → *Running* → *Completed*). Note the **time** from start to finish.

📸 **Screenshots for the class page:** the steps above, and the finished run. Before saving them, **paint out** your name, avatar, and any email address.

Then record what happened (change the result, minutes, and note):

In [ ]:
# Change the result, minutes, and note to what happened, then run
log_test("C4 Cromwell PLINK chr21 from the Workflows tab", "Cromwell server mode",
         result = "…", minutes = NA, note = "…")

If it succeeded, check its result against the app's. The results are in `<bucket>/<job name>/<workflow>/<job ID>/call-<task>/`: paste the **job name** from the **Job status** page here.

In [ ]:
JOB_NAME <- "plink_freq_job_…"   # paste the job name

tab_afreq <- system2("gcloud", c("storage", "ls", "--recursive",
                                 paste0(BUCKET_GS, "/", JOB_NAME, "/**chr.afreq")), stdout = TRUE)
tab_afreq
compare_freq(tab_afreq[1], "tab_chr21")

## Part 4. Nextflow

Nextflow runs from the app too (`nextflow run …`), and sends its tasks to Google Batch. The tutorial notes that the Nextflow **web interface** (Seqera) doesn't work in *All of Us*, because workspaces have no internet access, and that the app has **Nextflow 25.10** (version 26.04 has a stricter syntax).

### The configuration file

Nextflow's settings go in a `nextflow.config` file. Ours follows the *All of Us* tutorial, with four changes, marked in the comments.

⚠️ **Change 4 is essential.** Nextflow uses the `ps` command **inside the container** to measure each task's CPU and memory. The tutorial's default image (`ubuntu-slim`) doesn't have it, so our *hello* test failed with *"Command 'ps' required by nextflow to collect task metrics cannot be found"*. The official `ubuntu` image leaves it out too. We use the mirrored PLINK image, which has it. If you build your own image, install the `procps` package.

In [ ]:
# Nextflow copies files with a helper image. The tutorial uses the moving tag "cloud-sdk:alpine";
# we pin a version, because a new version of the same kind of image broke dsub in November 2026.
COPY_IMAGE <- "gcr.io/google.com/cloudsdktool/google-cloud-cli:591.0.0-alpine"

nextflow_conf <- file.path(TEST_DIR, "nextflow", "nextflow.config")
write_lines(glue::glue('
process.executor  = "google-batch"
process.container = "{PLINK_IMAGE}"   // change 4: the image must include ps (procps); ubuntu-slim in the tutorial does not
workDir           = "{WORK_GS}/nextflow-scratch"

google.location = "us-central1"   // change 1: no fixed zone, so Google can use any zone in the region
google.project  = "{BILLING_PROJECT}"
google.enableRequesterPaysBuckets = true

google.batch.serviceAccountEmail = "{SERVICE_ACCOUNT}"   // change 2: from gcloud, not PET_SA_EMAIL
google.batch.network           = "global/networks/network"
google.batch.subnetwork        = "regions/us-central1/subnetworks/subnetwork"
google.batch.usePrivateAddress = true
google.batch.copyImage         = "{COPY_IMAGE}"   // change 3: a pinned version
google.batch.bootDiskSize      = "20.GB"
'), nextflow_conf)

cat(str_replace(read_lines(nextflow_conf), SERVICE_ACCOUNT, "<your account>"), sep = "\n")

**🔍 Read the output:** the configuration file, with your account hidden. The settings match the table on the class page.

### A helper to run and time a Nextflow workflow

`run_nextflow()` runs a `.nf` file with our configuration, saves Nextflow's log, and writes a **trace** file: a table with one row per task, including its status and how long it took. If a run fails, it also prints the end of the failed task's **error file** (`.command.err`, in the task's work folder). We can't read Google's own job logs: *All of Us* blocks them (*"Permission denied for all log views"*).

In [ ]:
run_nextflow <- function(nf_file, name) {
  old <- setwd(file.path(TEST_DIR, "nextflow")); on.exit(setwd(old))
  trace_file <- paste0(name, "_trace.tsv")
  unlink(trace_file)
  seconds <- system.time(
    out <- suppressWarnings(system2("nextflow", c("-log", paste0(name, ".log"), "run", nf_file,
                                                  "-c", "nextflow.config", "-ansi-log", "false",
                                                  "-with-trace", trace_file),
                                    stdout = TRUE, stderr = TRUE))
  )[["elapsed"]]
  status <- if (is.null(attr(out, "status"))) "Succeeded" else "Failed"
  trace <- if (file.exists(trace_file)) read_tsv(trace_file, col_types = cols(.default = "c")) else tibble()

  message("Status: ", status, "   Minutes: ", round(seconds / 60, 1))
  if (status != "Succeeded") {
    all_lines <- c(out, read_lines(paste0(name, ".log")))
    print(error_lines(all_lines))
    # Cloud Logging is blocked in All of Us, so read the failed task's own error file
    work <- unique(na.omit(str_extract(all_lines, "gs://\\S+/nextflow-scratch/[0-9a-f]{2}/[0-9a-f]+")))
    for (w in head(work, 1)) {
      message("The task's error file (", w, "/.command.err) ends with:")
      print(tail(suppressWarnings(system2("gcloud", c("storage", "cat", paste0(w, "/.command.err")),
                                          stdout = TRUE, stderr = FALSE)), 10))
    }
  }
  invisible(list(status = status, minutes = seconds / 60, out = out, trace = trace))
}

### Test N1: *hello*

One task that prints a greeting. If this works, the **configuration** is right. A few minutes.

In [ ]:
write_lines('
process say_hello {
  output:
  stdout

  script:
  """
  echo "hello from Nextflow"
  """
}

workflow {
  say_hello() | view
}
', file.path(TEST_DIR, "nextflow", "hello.nf"))

n1 <- run_nextflow("hello.nf", "n1_hello")
str_subset(n1$out, "hello from Nextflow")
log_test("N1 Nextflow hello (pinned copy image, PLINK image)", "Nextflow", n1$status, n1$minutes)

**🔍 Read the output:** `Status: Succeeded` and the line `hello from Nextflow`.

If it **failed**, the lines printed after the status, and the end of the task's error file, are the likely cause. Common ones:

| Message | Likely cause | Try |
|---|---|---|
| *Command 'ps' required by nextflow to collect task metrics cannot be found* | the container has no `ps` | change 4: use an image that has it (the mirrored PLINK image does) |
| *Permission denied for all log views* | Nextflow tried to read Google's job logs, which *All of Us* blocks | nothing: read the task's `.command.err` instead (printed above) |
| a message about the **copy image**, or `exit status 127` | the helper image | try the tutorial's `COPY_IMAGE <- "gcr.io/google.com/cloudsdktool/cloud-sdk:alpine"`, rerun the configuration cell and this one |
| the image can't be **pulled** | the image isn't from the mirror | use the *All of Us* mirror (`MIRROR`), as in the dsub class |

Record what you saw and what you changed, e.g. `log_test("N1 …", "Nextflow", "Succeeded", n1$minutes, "needed …")`.

### Test N2: PLINK on chromosomes 19 to 22

The `.nf` file from the class page, plus three lines: `cpus` and `memory` (the machine size), and `publishDir`, which **copies** the results to one folder in the bucket, so they're easy to find. Nextflow starts **one task per chromosome**, all at the same time.

The tutorial says Nextflow, unlike Cromwell and dsub, **doesn't copy** `gs://` inputs onto the machine: it **mounts** the bucket and reads the files **as it goes**. It warns that this can be slow for PLINK. Our timings will show whether it is, for a simple `--freq`.

In [ ]:
write_lines(c(
  paste0('params.plinkDir = "', CLINVAR_GS, '"'),
  paste0('params.image    = "', PLINK_IMAGE, '"'),
  paste0('params.outDir   = "', WORK_GS, '/nextflow_results"'),
'
process plink_check {
  container params.image
  cpus 2
  memory "8 GB"
  publishDir params.outDir, mode: "copy"

  input:
  tuple val(chr), path(bed), path(bim), path(fam)

  output:
  path "chr${chr}.afreq"

  script:
  """
  plink2 --bed ${bed} --bim ${bim} --fam ${fam} --maf 0.01 --freq --out chr${chr}
  """
}

workflow {
  Channel.of(19..22)
    .map { c -> tuple(c, file("${params.plinkDir}/clinvar.chr${c}.bed"),
                         file("${params.plinkDir}/clinvar.chr${c}.bim"),
                         file("${params.plinkDir}/clinvar.chr${c}.fam")) }
    | plink_check
}
'), file.path(TEST_DIR, "nextflow", "plink_freq.nf"))

n2 <- run_nextflow("plink_freq.nf", "n2_plink_19_22")
n2$trace %>% select(any_of(c("name", "status", "exit", "submit", "duration", "realtime")))

**🔍 Read the output:** the status and total minutes, then the trace: one row per chromosome, with its status, **duration** (from submitting the task to the end, including starting the machine) and **realtime** (PLINK itself). The gap between the two is the time spent starting machines and reading files.

Now check chromosome 21 against the app's result:

In [ ]:
n2_check <- compare_freq(paste0(WORK_GS, "/nextflow_results/chr21.afreq"), "nextflow_chr21")
n2_check
log_test("N2 Nextflow PLINK chr19-22 (mirror image)", "Nextflow", n2$status, n2$minutes,
         paste0(nrow(n2$trace), " tasks; chr21 matched ", n2_check[["matched"]], "/",
                n2_check[["variants_app"]], "; task durations: ",
                paste(n2$trace$duration, collapse = ", ")))

**🔍 Read the output:** as for Cromwell: the same number of variants, all matched, and a difference close to 0.

## Part 4b. Optional tests on the Workflows tab

Three more ways to run a workflow, all **by hand** on the **Workflows** tab. Each takes about 10 minutes, mostly waiting. They need Part 1, the `compare_freq()` cell after C2, and, for N3, the Nextflow configuration cell. Record each with `log_test()`.

| Test | What it checks |
|---|---|
| **C5** A **batch job**: one row per chromosome (19 to 22) in an input table | the tab's version of dsub's `--tasks` |
| **C6** A WDL straight from our **GitHub** repository | whether the tab can reach GitHub from inside the perimeter |
| **N3** **Nextflow** from the tab | whether the tab runs Nextflow with our configuration |

### Test C5: a batch job from an input table

A batch job runs the same workflow **once per row** of a table. We make a table with one row per chromosome; the tab then asks which **column** goes with which **input**.

In [ ]:
batch_table <- tibble(
  chr          = 19:22,
  bed          = paste0(CLINVAR_GS, "/clinvar.chr", chr, ".bed"),
  bim          = paste0(CLINVAR_GS, "/clinvar.chr", chr, ".bim"),
  fam          = paste0(CLINVAR_GS, "/clinvar.chr", chr, ".fam"),
  docker_image = PLINK_IMAGE
)
batch_csv <- file.path(TEST_DIR, "batch_chr19_22.csv")
write_csv(batch_table, batch_csv)

TAB_GS <- paste0(WORK_GS, "/workflows_tab")
system2("gcloud", c("storage", "cp", batch_csv, paste0(TAB_GS, "/")))
batch_table %>% select(chr, bed)

**🔍 Read the output:** 4 rows, one per chromosome. (No participant data: only file addresses.) The table is saved in two places: `~/workflow_tests/batch_chr19_22.csv` in the app (download it from the file browser if the tab asks you to **upload** a file), and `data/workflows/workflows_tab/` in your bucket.

**On the Workflows tab:** **Setup** → `plink_freq` → **+ New job** → *Run a batch job with inputs defined by an input table*. Give it the table, then **match** the columns: `bed` → `plink_freq.bed`, and so on (leave `chr` unmatched; it's only a label). Outputs: your bucket. **Submit job**, and watch **Job status**.

📸 Screenshots: the input table step, the column matching, and the finished batch. Note how the table is given (upload or bucket), how the batch shows on **Job status**, and the minutes from start to finish.

Then check chromosome 21 against the app. Paste the **job name** from **Job status**:

In [ ]:
BATCH_JOB <- "plink_freq_job_…"   # paste the job name

batch_files <- system2("gcloud", c("storage", "ls", "--recursive",
                                   paste0(BUCKET_GS, "/", BATCH_JOB, "/**chr.afreq")), stdout = TRUE)
length(batch_files)   # 4: one per row

# Which file is chromosome 21? Read the first lines of each (the first column is the chromosome)
first_chrom <- function(f) {
  top <- system2("gcloud", c("storage", "cat", "--range", "0-300", f), stdout = TRUE)
  str_extract(top[2], "^[^\t]+")
}
batch_chroms <- map_chr(batch_files, first_chrom)
batch_chroms

c5_check <- compare_freq(batch_files[str_remove(batch_chroms, "^chr") == "21"][1], "batch_chr21")
c5_check

In [ ]:
# Change the minutes and note to what happened, then run
log_test("C5 Cromwell batch job, chr19-22, from the Workflows tab", "Cromwell server mode",
         result = "…", minutes = NA,
         note = paste0("table given as: …; Job status shows: …; chr21 matched ",
                       c5_check[["matched"]], "/", c5_check[["variants_app"]]))

**🔍 Read the output:** 4 results files, one per chromosome (`19` to `22`), and chromosome 21 matching the app, as in C4.

### Test C6: a WDL from GitHub

The same WDL is in our GitHub repository: [`notebooks/workflows/wdl/plink_freq.wdl`](https://github.com/CWML/All-of-Us-Training-Resources/blob/main/notebooks/workflows/wdl/plink_freq.wdl). *All of Us* workspaces sit inside a **perimeter** that blocks most of the internet, so this tests whether the tab can still fetch a workflow from GitHub.

**On the Workflows tab:** **Setup** → **+ Add workflow** → **Git repository**. Give the repository address, `https://github.com/CWML/All-of-Us-Training-Resources.git`, and, if asked, the branch (`main`) and the path (`notebooks/workflows/wdl/plink_freq.wdl`). Note **every** field it asks for, and any error. If it adds the workflow, run a **single job** with the same inputs as C4 (printed below), and compare as in C4.

In [ ]:
cat(read_lines(file.path(TEST_DIR, "cromwell", "plink_freq.json")), sep = "\n")   # the values for the form

# After the run, paste the job name and compare:
# C6_JOB <- "…"
# c6_file <- system2("gcloud", c("storage", "ls", "--recursive", paste0(BUCKET_GS, "/", C6_JOB, "/**chr.afreq")), stdout = TRUE)
# compare_freq(c6_file[1], "git_chr21")

log_test("C6 Cromwell WDL from GitHub, on the Workflows tab", "Cromwell server mode",
         result = "…", minutes = NA, note = "fields asked: …; error (if any): …")

**🔍 Read the output:** the four input values. If adding the workflow **failed**, the error message is the result: write it in the note. That tells the class whether to use Git sources or copy WDL files into the bucket.

### Test N3: Nextflow from the Workflows tab

The tab can run Nextflow too: it needs the `.nf` file with a `nextflow.config` **next to it** in the bucket. We copy both, with the results going to a separate folder (`nextflow_results_tab`), so we can tell them apart from N2's.

In [ ]:
NF_TAB_GS <- paste0(WORK_GS, "/workflows_tab/nextflow")
nf_tab <- file.path(TEST_DIR, "nextflow", "tab")
dir.create(nf_tab, showWarnings = FALSE)

write_lines(str_replace(read_lines(file.path(TEST_DIR, "nextflow", "plink_freq.nf")),
                        "/nextflow_results\"", "/nextflow_results_tab\""),
            file.path(nf_tab, "plink_freq.nf"))
file.copy(nextflow_conf, file.path(nf_tab, "nextflow.config"), overwrite = TRUE)

system2("gcloud", c("storage", "cp", file.path(nf_tab, c("plink_freq.nf", "nextflow.config")),
                    paste0(NF_TAB_GS, "/")))
system2("gcloud", c("storage", "ls", paste0(NF_TAB_GS, "/")), stdout = TRUE)

**🔍 Read the output:** two files in `data/workflows/workflows_tab/nextflow/`.

**On the Workflows tab:** **+ Add workflow** → your workspace bucket → `data/workflows/workflows_tab/nextflow/plink_freq.nf` → add it → **+ New job** → a single job. Note what the **inputs** step shows (the `params`?) and the **outputs** step, then submit. If it fails, note the message on **Job status**.

Then compare chromosome 21:

In [ ]:
n3_check <- compare_freq(paste0(WORK_GS, "/nextflow_results_tab/chr21.afreq"), "nextflow_tab_chr21")
n3_check

# Change the result, minutes, and note to what happened, then run
log_test("N3 Nextflow from the Workflows tab", "Nextflow server mode",
         result = "…", minutes = NA,
         note = paste0("inputs step showed: …; chr21 matched ", n3_check[["matched"]], "/", n3_check[["variants_app"]]))

**🔍 Read the output:** chromosome 21 matching the app. If there's **no results file**, look in the job's folder in your bucket: the tab may ignore `publishDir` and keep results in its own folder (`gcloud storage ls --recursive gs://…/<job name>/**chr21.afreq`).

## Part 5. The test log, and cleaning up

### What to report back

The table below is everything the class page needs. It has **no participant data**: test names, minutes, and counts of **variants**. Copy it (and the tool versions from Part 1) into the conversation with your AI assistant, or into your notes.

In [ ]:
options(width = 200)
test_log
write_csv(test_log, file.path(TEST_DIR, "workflow_test_log.csv"))

**🔍 Read the output:** one row per test. For any that **failed**, the note has the first error lines; if you fixed it, add a row that says how.

### Clean up

Both tools leave **work files** in the bucket (copies of inputs, logs, scripts). They cost a little to store. Delete them when the tests are done (uncomment to run):

In [ ]:
# system2("gcloud", c("storage", "rm", "--recursive", paste0(WORK_GS, "/cromwell-executions")))
# system2("gcloud", c("storage", "rm", "--recursive", paste0(WORK_GS, "/nextflow-scratch")))